# ANTON — Notebook 02: S11, VSWR, and Input Impedance Analysis

Visualizes frequency responses, detects resonant frequencies, computes -10 dB bandwidth, and analyzes complex input impedance Zin.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from anton.datasets.storage import DatasetStorageManager
from anton.datasets.normalization import detect_resonances_and_bandwidth

storage = DatasetStorageManager()
records = storage.db.list_records(limit=1)

if not records:
    print("[EMPTY STATE] No simulation records available for visualization.")
else:
    rec_id = records[0]['record_id']
    arrays = storage.arrays.load_record_arrays(rec_id)
    freqs = arrays['frequencies_ghz']
    s11_db = arrays.get('s11_db')
    vswr = arrays.get('vswr')

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    if s11_db is not None:
        ax1.plot(freqs, s11_db, 'b-', lw=2, label='S11 (dB)')
        ax1.axhline(-10, color='r', linestyle='--', label='-10 dB Threshold')
        ax1.set_xlabel('Frequency (GHz)')
        ax1.set_ylabel('S11 (dB)')
        ax1.set_title(f'Return Loss — {rec_id}')
        ax1.grid(True)
        ax1.legend()

    if vswr is not None:
        ax2.plot(freqs, vswr, 'g-', lw=2, label='VSWR')
        ax2.axhline(2.0, color='r', linestyle='--', label='VSWR = 2.0')
        ax2.set_xlabel('Frequency (GHz)')
        ax2.set_ylabel('VSWR')
        ax2.set_title('Voltage Standing Wave Ratio')
        ax2.set_ylim(1.0, 10.0)
        ax2.grid(True)
        ax2.legend()

    plt.tight_layout()
    plt.show()


In [ ]:
# Complex Input Impedance Analysis (Rin + j Xin)
if records:
    rec_id = records[0]['record_id']
    arrays = storage.arrays.load_record_arrays(rec_id)
    zin_re = arrays.get('zin_real')
    zin_im = arrays.get('zin_imag')

    if zin_re is not None and zin_im is not None:
        plt.figure(figsize=(8, 5))
        plt.plot(arrays['frequencies_ghz'], zin_re, 'r-', lw=2, label='R_in (Resistance, Ohm)')
        plt.plot(arrays['frequencies_ghz'], zin_im, 'b--', lw=2, label='X_in (Reactance, Ohm)')
        plt.axhline(50, color='k', linestyle=':', label='Z0 = 50 Ohm')
        plt.axhline(0, color='gray', linestyle=':')
        plt.xlabel('Frequency (GHz)')
        plt.ylabel('Impedance (Ohm)')
        plt.title('Complex Input Impedance')
        plt.grid(True)
        plt.legend()
        plt.show()
    else:
        print("[INFO] Record contains magnitude-only S11. Zin is not calculated to prevent fabricated phase.")
